# Part 2. Rain API (Open-Meteo)
**Group 6 - rainy season January 1 to May 31, 2023**

## Step 1. Read the departments table from Wikipedia
Wikipedia rejects direct `pd.read_html(url)` (HTTP 403), so the page is fetched
with a browser `User-Agent` first and parsed from the response text. The page
"Anexo:Departamentos del Perú" lists the 24 departments; only the columns
departamento and capital are kept.

In [1]:
import io
import time
import unicodedata
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display

BASE = Path.cwd()
if (BASE / "assignment_2").is_dir():
    BASE = BASE / "assignment_2"
DATOS = BASE / "datos"
DATOS.mkdir(parents=True, exist_ok=True)

URL_WIKI = "https://es.wikipedia.org/wiki/Anexo:Departamentos_del_Per%C3%BA"
respuesta = requests.get(URL_WIKI, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
respuesta.raise_for_status()
tablas = pd.read_html(io.StringIO(respuesta.text))

# The departments table is the one with Departamento and Capital columns.
tabla = next(t for t in tablas
             if {"Departamento", "Capital"}.issubset(t.columns))
departamentos = tabla[["Departamento", "Capital"]].copy()
departamentos.columns = ["departamento", "capital"]
print("Rows read:", len(departamentos))

Rows read: 24


## Step 2. Verify the 25 departments and clean the capitals
The table has 24 departments; Callao is missing because it is a constitutional
province, not a department. It is taken from the special-regime provinces table
on the same Wikipedia page, which also lists Callao with capital Callao.

Lima's capital appears as "Huacho (de facto)": the parenthetical is stripped
and we geocode **Huacho**, the de facto capital listed by Wikipedia (the city
that hosts the departmental government of Lima). Any other extra text between
parentheses is removed the same way before calling the API.

In [2]:
assert len(departamentos) == 24
assert "Callao" not in departamentos["departamento"].values

provincias = next(t for t in tablas if {"Provincia", "Capital"}.issubset(t.columns))
fila_callao = provincias.loc[
    provincias["Provincia"].str.replace(r"\[.*?\]", "", regex=True).str.strip()
    .eq("Callao"), "Capital"].iloc[0]
departamentos = pd.concat(
    [departamentos, pd.DataFrame({"departamento": ["Callao"],
                                  "capital": [fila_callao]})],
    ignore_index=True)

# Clean extra text in capital cells (e.g. "Huacho (de facto)").
departamentos["capital"] = (departamentos["capital"]
                            .str.replace(r"\(.*?\)", "", regex=True)
                            .str.strip())
assert len(departamentos) == 25
assert departamentos["departamento"].is_unique
assert departamentos["capital"].notna().all()
display(departamentos)

,departamento,capital
0,Amazonas,Chachapoyas
1,Áncash,Huaraz
2,Apurímac,Abancay
3,Arequipa,Arequipa
4,Ayacucho,Ayacucho
5,Cajamarca,Cajamarca
6,Cusco,Cusco
7,Huancavelica,Huancavelica
8,Huánuco,Huánuco
9,Ica,Ica


## Step 3. Geocode each capital and verify the department
The geocoding API can return same-named cities outside Peru, so for each capital
the first result with `country_code == "PE"` is taken. The API's `admin1` is not
written consistently ("Departamento de Cusco", "Ancash" without tilde,
"Provincia Constitucional del Callao"), so it is normalized (prefixes stripped,
accents removed, lowercase) before comparing with the expected department. Any
mismatch is flagged row by row.

In [3]:
def normalizar(texto):
    texto = unicodedata.normalize("NFD", str(texto))
    texto = "".join(c for c in texto if unicodedata.category(c) != "Mn")
    texto = texto.lower().strip()
    for prefijo in ("departamento de ", "provincia constitucional del ",
                    "provincia de ", "region ", "región "):
        if texto.startswith(prefijo):
            texto = texto[len(prefijo):]
    return texto

def geocodificar(ciudad):
    r = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                     params={"name": ciudad, "count": 10, "language": "es"},
                     timeout=60)
    r.raise_for_status()
    time.sleep(1.1)  # At least one second between requests.
    return [x for x in r.json().get("results", []) if x.get("country_code") == "PE"]

filas_geo = []
for _, fila in departamentos.iterrows():
    candidatos = geocodificar(fila["capital"])
    assert candidatos, f"No Peruvian result for {fila['capital']}"
    esperado = normalizar(fila["departamento"])
    elegido = next((c for c in candidatos
                    if normalizar(c.get("admin1", "")) == esperado),
                   candidatos[0])
    filas_geo.append({
        "departamento": fila["departamento"],
        "capital": fila["capital"],
        "admin1": elegido.get("admin1"),
        "latitud": elegido["latitude"],
        "longitud": elegido["longitude"],
        "admin1_ok": normalizar(elegido.get("admin1", "")) == esperado,
    })

geocodificacion = pd.DataFrame(filas_geo)
display(geocodificacion)
assert geocodificacion["admin1_ok"].all(), "Mismatched admin1 rows must be reviewed"
print("All 25 rows are in Peru and in the correct department.")

,departamento,capital,admin1,latitud,longitud,admin1_ok
0,Amazonas,Chachapoyas,Departamento de Amazonas,-6.23169,-77.86903,True
1,Áncash,Huaraz,Ancash,-9.52614,-77.52869,True
2,Apurímac,Abancay,Apurímac,-13.63426,-72.88380,True
3,Arequipa,Arequipa,Departamento de Arequipa,-16.39899,-71.53747,True
4,Ayacucho,Ayacucho,Departamento de Ayacucho,-13.16380,-74.22345,True
5,Cajamarca,Cajamarca,Departamento de Cajamarca,-7.16378,-78.50027,True
6,Cusco,Cusco,Departamento de Cusco,-13.53188,-71.96701,True
7,Huancavelica,Huancavelica,Departamento de Huancavelica,-12.78691,-74.97298,True
8,Huánuco,Huánuco,Departamento de Huánuco,-9.92882,-76.23989,True
9,Ica,Ica,Departamento de Ica,-14.07538,-75.73422,True


All 25 rows are in Peru and in the correct department.


## Step 4. Daily precipitation for the season
Daily precipitation is requested from the Open-Meteo archive for each capital,
from 2023-01-01 to 2023-05-31, in the America/Lima timezone. Per department we
compute `lluvia_total_mm` (sum of daily rain) and `dias_lluvia_fuerte` (days
with at least 20 mm).

In [4]:
def lluvia_diaria(latitud, longitud):
    r = requests.get("https://archive-api.open-meteo.com/v1/archive",
                     params={"latitude": latitud, "longitude": longitud,
                             "start_date": "2023-01-01", "end_date": "2023-05-31",
                             "daily": "precipitation_sum",
                             "timezone": "America/Lima"},
                     timeout=60)
    r.raise_for_status()
    time.sleep(1.1)  # At least one second between requests.
    return pd.Series(r.json()["daily"]["precipitation_sum"],
                     index=pd.to_datetime(r.json()["daily"]["time"]))

serie_diaria = {}
resumen = []
for _, fila in geocodificacion.iterrows():
    serie = lluvia_diaria(fila["latitud"], fila["longitud"])
    serie_diaria[fila["departamento"]] = serie
    resumen.append({
        "departamento": fila["departamento"],
        "capital": fila["capital"],
        "latitud": fila["latitud"],
        "longitud": fila["longitud"],
        "lluvia_total_mm": round(float(serie.sum(skipna=True)), 1),
        "dias_lluvia_fuerte": int((serie >= 20).sum()),
    })

lluvias = pd.DataFrame(resumen)
assert len(lluvias) == 25
display(lluvias)

,departamento,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte
0,Amazonas,Chachapoyas,-6.23169,-77.86903,414.6,3
1,Áncash,Huaraz,-9.52614,-77.52869,1145.8,9
2,Apurímac,Abancay,-13.63426,-72.88380,726.2,3
3,Arequipa,Arequipa,-16.39899,-71.53747,362.0,1
4,Ayacucho,Ayacucho,-13.16380,-74.22345,498.4,2
5,Cajamarca,Cajamarca,-7.16378,-78.50027,796.6,0
6,Cusco,Cusco,-13.53188,-71.96701,714.0,1
7,Huancavelica,Huancavelica,-12.78691,-74.97298,817.8,3
8,Huánuco,Huánuco,-9.92882,-76.23989,524.3,2
9,Ica,Ica,-14.07538,-75.73422,113.6,0


## Step 5. Missing-data check
Before trusting the totals, the number of missing (None) daily values is counted
per department. 0 is a valid answer, but it must be verified, not assumed.

In [5]:
faltantes = pd.Series({dep: int(s.isna().sum()) for dep, s in serie_diaria.items()})
print("Expected days in the season:",
      (pd.Timestamp("2023-05-31") - pd.Timestamp("2023-01-01")).days + 1)
print("Days returned per department:", {dep: len(s) for dep, s in list(serie_diaria.items())[:3]}, "...")
print("Missing days per department:")
print(faltantes.to_string())
assert len({len(s) for s in serie_diaria.values()}) == 1
assert faltantes.sum() == 0, "Missing days would have to be handled explicitly"
print("No missing days: totals and heavy-rain counts use every day of the season.")

Expected days in the season: 151
Days returned per department: {'Amazonas': 151, 'Áncash': 151, 'Apurímac': 151} ...
Missing days per department:
Amazonas         0
Áncash           0
Apurímac         0
Arequipa         0
Ayacucho         0
Cajamarca        0
Cusco            0
Huancavelica     0
Huánuco          0
Ica              0
Junín            0
La Libertad      0
Lambayeque       0
Lima             0
Loreto           0
Madre de Dios    0
Moquegua         0
Pasco            0
Piura            0
Puno             0
San Martín       0
Tacna            0
Tumbes           0
Ucayali          0
Callao           0
No missing days: totals and heavy-rain counts use every day of the season.


In [6]:
ruta = DATOS / "lluvias_por_departamento.csv"
lluvias.to_csv(ruta, index=False, encoding="utf-8-sig")
print("Saved:", ruta)
verificacion = pd.read_csv(ruta)
assert list(verificacion.columns) == ["departamento", "capital", "latitud",
                                      "longitud", "lluvia_total_mm", "dias_lluvia_fuerte"]
assert len(verificacion) == 25
display(verificacion)

Saved: C:\Users\USER\Documents\GitHub\PythonGroup06\assignment_2\datos\lluvias_por_departamento.csv


,departamento,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte
0,Amazonas,Chachapoyas,-6.23169,-77.86903,414.6,3
1,Áncash,Huaraz,-9.52614,-77.52869,1145.8,9
2,Apurímac,Abancay,-13.63426,-72.88380,726.2,3
3,Arequipa,Arequipa,-16.39899,-71.53747,362.0,1
4,Ayacucho,Ayacucho,-13.16380,-74.22345,498.4,2
5,Cajamarca,Cajamarca,-7.16378,-78.50027,796.6,0
6,Cusco,Cusco,-13.53188,-71.96701,714.0,1
7,Huancavelica,Huancavelica,-12.78691,-74.97298,817.8,3
8,Huánuco,Huánuco,-9.92882,-76.23989,524.3,2
9,Ica,Ica,-14.07538,-75.73422,113.6,0
